# Exploratory Data Analysis

**Project:** Student Retention & Welfare Tracker  
**Role:** Person 2 — Data Analysis & Data Science

This notebook explores the cleaned datasets to identify patterns and
relationships between student attendance, Mid-Day Meal utilization,
school infrastructure, and student test performance.

The analysis will focus on:
- Attendance patterns and anomalies
- MDM procurement and utilization
- School infrastructure conditions
- Student test-score performance
- District-level patterns
- Relationships between welfare factors and attendance/performance

In [1]:
import pandas as pd

attendance = pd.read_csv("../data/processed/attendance_clean.csv")
infrastructure = pd.read_csv("../data/processed/infrastructure_clean.csv")
mdm = pd.read_csv("../data/processed/mid_day_meal_procurement_cleaned.csv")
school_master = pd.read_csv("../data/processed/school_master_cleaned.csv")
test_scores = pd.read_csv("../data/processed/test_scores_clean.csv")

datasets = {
    "Attendance": attendance,
    "Infrastructure": infrastructure,
    "MDM": mdm,
    "School Master": school_master,
    "Test Scores": test_scores
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", list(df.columns))


Attendance
----------
Rows: 20000
Columns: 13
Column names: ['record_id', 'grade', 'total_students', 'present_students', 'marked_by', 'attendance_count_anomaly', 'attendance_rate', 'date', 'day_of_week', 'is_sunday', 'proxy_attendance_flag', 'school_id', 'teacher_present']

Infrastructure
--------------
Rows: 3000
Columns: 10
Column names: ['inspection_id', 'has_electricity', 'has_drinking_water', 'has_functional_toilet', 'has_boundary_wall', 'has_playground', 'inspector_name', 'remarks', 'date', 'school_id']

MDM
---
Rows: 12000
Columns: 9
Column names: ['procurement_id', 'date', 'school_id', 'vendor_name', 'grain_type', 'quantity', 'unit', 'total_cost', 'payment_status']

School Master
-------------
Rows: 600
Columns: 7
Column names: ['school_id', 'school_name', 'district', 'block', 'total_enrolled_students', 'school_type', 'medium']

Test Scores
-----------
Rows: 8000
Columns: 10
Column names: ['assessment_id', 'grade', 'grading_scale', 'avg_score', 'max_marks', 'total_students_ass

In [2]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("=" * len(name))
    print(df.isna().sum())
    


Attendance
record_id                    405
grade                          0
total_students                 0
present_students               0
marked_by                   3369
attendance_count_anomaly       0
attendance_rate                0
date                           0
day_of_week                    0
is_sunday                      0
proxy_attendance_flag          0
school_id                      0
teacher_present             1169
dtype: int64

Infrastructure
inspection_id              0
has_electricity          209
has_drinking_water       179
has_functional_toilet    173
has_boundary_wall        253
has_playground           252
inspector_name           323
remarks                  488
date                       0
school_id                  0
dtype: int64

MDM
===
procurement_id    0
date              0
school_id         0
vendor_name       0
grain_type        0
quantity          0
unit              0
total_cost        0
payment_status    0
dtype: int64

School Master
school_id 

In [3]:
missing_summary = []

for name, df in datasets.items():
    total_missing = df.isna().sum().sum()
    rows_with_missing = df.isna().any(axis=1).sum()

    missing_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Total Missing Values": total_missing,
        "Rows With Missing Values": rows_with_missing
    })

missing_summary = pd.DataFrame(missing_summary)

missing_summary

,Dataset,Rows,Total Missing Values,Rows With Missing Values
0,Attendance,20000,4943,4673
1,Infrastructure,3000,1877,1432
2,MDM,12000,0,0
3,School Master,600,0,0
4,Test Scores,8000,1983,1983


In [4]:
for name, df in datasets.items():
    print(f"{name}: {df['school_id'].nunique()} unique school IDs")

Attendance: 600 unique school IDs
Infrastructure: 598 unique school IDs
MDM: 600 unique school IDs
School Master: 600 unique school IDs
Test Scores: 600 unique school IDs


In [5]:
school_id_sets = {
    name: set(df["school_id"].dropna().unique())
    for name, df in datasets.items()
}

for name, ids in school_id_sets.items():
    missing_from_master = ids - school_id_sets["School Master"]
    print(
        f"{name}: {len(missing_from_master)} school IDs "
        "not found in School Master"
    )

Attendance: 0 school IDs not found in School Master
Infrastructure: 0 school IDs not found in School Master
MDM: 0 school IDs not found in School Master
School Master: 0 school IDs not found in School Master
Test Scores: 0 school IDs not found in School Master


In [6]:
for name, df in datasets.items():
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        
        print(f"{name}")
        print(f"  Start date: {df['date'].min().date()}")
        print(f"  End date:   {df['date'].max().date()}")
        print()

Attendance
  Start date: 2025-01-04
  End date:   2026-12-03

Infrastructure
  Start date: 2025-01-04
  End date:   2026-12-03

MDM
  Start date: 2025-01-04
  End date:   2026-12-03

Test Scores
  Start date: 2025-01-04
  End date:   2026-12-03



In [7]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    
    school_counts = df.groupby("school_id").size()
    
    print("Minimum records per school:", school_counts.min())
    print("Maximum records per school:", school_counts.max())
    print("Average records per school:", round(school_counts.mean(), 2))


Attendance
----------
Minimum records per school: 16
Maximum records per school: 51
Average records per school: 33.33

Infrastructure
--------------
Minimum records per school: 1
Maximum records per school: 11
Average records per school: 5.02

MDM
---
Minimum records per school: 8
Maximum records per school: 33
Average records per school: 20.0

School Master
-------------
Minimum records per school: 1
Maximum records per school: 1
Average records per school: 1.0

Test Scores
-----------
Minimum records per school: 1
Maximum records per school: 28
Average records per school: 13.33


In [8]:
school_counts = test_scores.groupby("school_id").size()

print("Test Scores")
print("Minimum records per school:", school_counts.min())
print("Maximum records per school:", school_counts.max())
print("Average records per school:", round(school_counts.mean(), 2))


Test Scores
Minimum records per school: 1
Maximum records per school: 28
Average records per school: 13.33


## 1. School-Level Attendance Analysis

Attendance data contains multiple daily records for each school. To avoid
duplicating records when combining datasets, we first aggregate attendance
information to the school level.

The following metrics summarize attendance performance and data-quality
anomalies for each school:

- Average attendance rate
- Number of attendance records
- Number and rate of potential proxy attendance records
- Number and rate of attendance count anomalies
- Average total students
- Average present students

Anomaly records are retained rather than removed because they may provide
useful evidence of unusual attendance reporting.

### Attendance Record Coverage

The number of attendance observations should include every valid attendance
row, including records where the original `record_id` is missing.

Therefore, the attendance record count is calculated using `size` rather than
counting the `record_id` column. This prevents missing administrative IDs from
being excluded from attendance coverage calculations.

In [9]:
attendance_kpi = (
    attendance
    .groupby("school_id")
    .agg(
        avg_attendance_rate=("attendance_rate", "mean"),
        total_attendance_records=("record_id", "count"),
        proxy_attendance_records=("proxy_attendance_flag", "sum"),
        count_anomaly_records=("attendance_count_anomaly", "sum"),
        avg_total_students=("total_students", "mean"),
        avg_present_students=("present_students", "mean")
    )
    .reset_index()
)

attendance_kpi["proxy_attendance_rate"] = (
    attendance_kpi["proxy_attendance_records"]
    / attendance_kpi["total_attendance_records"]
    * 100
)

attendance_kpi["count_anomaly_rate"] = (
    attendance_kpi["count_anomaly_records"]
    / attendance_kpi["total_attendance_records"]
    * 100
)

attendance_kpi.head()

,school_id,avg_attendance_rate,total_attendance_records,proxy_attendance_records,count_anomaly_records,avg_total_students,avg_present_students,proxy_attendance_rate,count_anomaly_rate
0,SCH0001,82.288549,33,1,0,168.878788,138.090909,3.030303,0.000000
1,SCH0002,83.668212,34,3,3,171.314286,144.628571,8.823529,8.823529
2,SCH0003,80.746589,32,1,1,181.250000,147.500000,3.125000,3.125000
3,SCH0004,79.194911,29,1,3,151.137931,120.931034,3.448276,10.344828
4,SCH0005,80.676448,27,0,0,155.185185,127.444444,0.000000,0.000000


### Attendance KPI Validation

The attendance summary should contain exactly one row for each school.

We also validate the calculated rates to ensure that:
- Every school has a positive number of attendance observations.
- Proxy attendance rates remain between 0% and 100%.
- Count anomaly rates remain between 0% and 100%.
- School IDs remain complete.

These checks help ensure that the aggregated attendance metrics are reliable
before they are combined with other school-level datasets.

In [10]:
print("Rows in attendance KPI table:", len(attendance_kpi))
print("Unique school IDs:", attendance_kpi["school_id"].nunique())

print(
    "Duplicate school IDs:",
    attendance_kpi["school_id"].duplicated().sum()
)

print(
    "Schools with zero attendance records:",
    (attendance_kpi["total_attendance_records"] == 0).sum()
)

print(
    "Proxy attendance rates outside 0-100%:",
    (
        (attendance_kpi["proxy_attendance_rate"] < 0) |
        (attendance_kpi["proxy_attendance_rate"] > 100)
    ).sum()
)

print(
    "Count anomaly rates outside 0-100%:",
    (
        (attendance_kpi["count_anomaly_rate"] < 0) |
        (attendance_kpi["count_anomaly_rate"] > 100)
    ).sum()
)

print(
    "Missing school IDs:",
    attendance_kpi["school_id"].isna().sum()
)

Rows in attendance KPI table: 600
Unique school IDs: 600
Duplicate school IDs: 0
Schools with zero attendance records: 0
Proxy attendance rates outside 0-100%: 0
Count anomaly rates outside 0-100%: 0
Missing school IDs: 0


## 2. School-Level MDM Analysis

The Mid-Day Meal (MDM) dataset contains multiple procurement records for each
school. To prevent row multiplication when combining MDM with other datasets,
we aggregate procurement information to the school level.

The following metrics summarize MDM procurement activity:

- Total quantity of grain procured
- Number of procurement records
- Total procurement cost
- Average procurement cost per record
- Number of unique vendors
- Number of unique grain types
- Payment status distribution

The cleaned MDM dataset has already standardized grain types, quantities, units,
and costs. These school-level metrics will later be used to investigate
relationships between MDM regularity, attendance, and school welfare outcomes.

In [11]:
mdm_kpi = (
    mdm
    .groupby("school_id")
    .agg(
        total_mdm_quantity_kg=("quantity", "sum"),
        total_mdm_records=("school_id", "size"),
        total_mdm_cost=("total_cost", "sum"),
        avg_mdm_cost=("total_cost", "mean"),
        unique_vendors=("vendor_name", "nunique"),
        unique_grain_types=("grain_type", "nunique")
    )
    .reset_index()
)

mdm_kpi.head()

,school_id,total_mdm_quantity_kg,total_mdm_records,total_mdm_cost,avg_mdm_cost,unique_vendors,unique_grain_types
0,SCH0001,674.1,19,32891.0,1731.105263,9,7
1,SCH0002,542.8,16,29130.0,1820.625000,9,6
2,SCH0003,529.8,16,32771.0,2048.187500,9,5
3,SCH0004,567.2,16,38586.0,2411.625000,8,6
4,SCH0005,796.7,23,42220.0,1835.652174,11,7


### MDM KPI Validation

The MDM summary should contain exactly one row for each school.

We validate that:
- Every school has at least one procurement record.
- Total quantity is non-negative.
- Total procurement cost is non-negative.
- No school IDs are missing or duplicated.
- The aggregated MDM table contains all 600 schools.

These checks ensure that procurement metrics are safe to combine with other
school-level datasets.

In [12]:
print("Rows in MDM KPI table:", len(mdm_kpi))
print("Unique school IDs:", mdm_kpi["school_id"].nunique())

print(
    "Duplicate school IDs:",
    mdm_kpi["school_id"].duplicated().sum()
)

print(
    "Schools with zero procurement records:",
    (mdm_kpi["total_mdm_records"] == 0).sum()
)

print(
    "Negative MDM quantities:",
    (mdm_kpi["total_mdm_quantity_kg"] < 0).sum()
)

print(
    "Negative MDM costs:",
    (mdm_kpi["total_mdm_cost"] < 0).sum()
)

print(
    "Missing school IDs:",
    mdm_kpi["school_id"].isna().sum()
)

Rows in MDM KPI table: 600
Unique school IDs: 600
Duplicate school IDs: 0
Schools with zero procurement records: 0
Negative MDM quantities: 0
Negative MDM costs: 0
Missing school IDs: 0


## 3. MDM Procurement Regularity

The dataset does not contain separate fields for MDM quantity consumed or
wasted. Therefore, actual MDM utilization and wastage cannot be calculated
directly.

As a defensible alternative, we measure MDM procurement regularity based on
the number of procurement records observed for each school during the study
period.

Higher procurement regularity indicates more consistent recorded MDM
procurement activity.

In [18]:
print("MDM procurement records per school:")
print(mdm_kpi["total_mdm_records"].describe())

print("\nSchools with the fewest procurement records:")
display(
    mdm_kpi[
        ["school_id", "total_mdm_records"]
    ]
    .sort_values("total_mdm_records")
    .head(10)
)

print("\nSchools with the most procurement records:")
display(
    mdm_kpi[
        ["school_id", "total_mdm_records"]
    ]
    .sort_values("total_mdm_records", ascending=False)
    .head(10)
)

MDM procurement records per school:
count    600.0000
mean      20.0000
std        4.3881
min        8.0000
25%       17.0000
50%       20.0000
75%       23.0000
max       33.0000
Name: total_mdm_records, dtype: float64

Schools with the fewest procurement records:


,school_id,total_mdm_records
107,SCH0108,8
452,SCH0453,9
156,SCH0157,9
399,SCH0400,10
172,SCH0173,10
38,SCH0039,10
45,SCH0046,10
304,SCH0305,10
34,SCH0035,10
198,SCH0199,11



Schools with the most procurement records:


,school_id,total_mdm_records
225,SCH0226,33
209,SCH0210,33
385,SCH0386,32
530,SCH0531,32
322,SCH0323,31
228,SCH0229,31
282,SCH0283,31
497,SCH0498,31
140,SCH0141,31
552,SCH0553,30


In [19]:
print("MDM procurement record distribution:")

display(
    mdm_kpi["total_mdm_records"]
    .value_counts()
    .sort_index()
)

MDM procurement record distribution:


total_mdm_records
8      1
9      2
10     6
11     5
12     8
13    13
14    22
15    25
16    46
17    47
18    58
19    55
20    52
21    46
22    48
23    41
24    33
25    28
26    16
27    18
28    10
29     7
30     4
31     5
32     2
33     2
Name: count, dtype: int64

### 3.2 Relative MDM Regularity Rate

Because the dataset does not provide an official expected MDM procurement
schedule, a fixed compliance threshold cannot be established from the data.

We therefore calculate a relative MDM Regularity Rate by comparing each
school's number of procurement records with the maximum number of procurement
records observed for any school.

This metric is intended for relative comparison between schools rather than
as a measure of statutory or operational compliance.

In [20]:
max_mdm_records = mdm_kpi["total_mdm_records"].max()

mdm_kpi["mdm_regularity_rate"] = (
    mdm_kpi["total_mdm_records"] / max_mdm_records
) * 100

print("Maximum MDM records for any school:", max_mdm_records)

display(
    mdm_kpi[
        ["school_id", "total_mdm_records", "mdm_regularity_rate"]
    ]
    .sort_values("mdm_regularity_rate")
    .head(10)
)

Maximum MDM records for any school: 33


,school_id,total_mdm_records,mdm_regularity_rate
107,SCH0108,8,24.242424
452,SCH0453,9,27.272727
156,SCH0157,9,27.272727
399,SCH0400,10,30.303030
172,SCH0173,10,30.303030
38,SCH0039,10,30.303030
45,SCH0046,10,30.303030
304,SCH0305,10,30.303030
34,SCH0035,10,30.303030
198,SCH0199,11,33.333333


### 3.3 MDM Regularity Rate Validation

The MDM Regularity Rate is validated to ensure that the calculated values
remain within the expected 0–100% range and that every school has a valid
regularity value.

This confirms that the metric can safely be used in school-level analysis
and later joined with other school-level KPIs.

In [21]:
print("Schools:", len(mdm_kpi))
print("Unique school IDs:", mdm_kpi["school_id"].nunique())

print(
    "Regularity rates below 0%:",
    (mdm_kpi["mdm_regularity_rate"] < 0).sum()
)

print(
    "Regularity rates above 100%:",
    (mdm_kpi["mdm_regularity_rate"] > 100).sum()
)

print(
    "Missing regularity rates:",
    mdm_kpi["mdm_regularity_rate"].isna().sum()
)

print(
    "Minimum regularity rate:",
    mdm_kpi["mdm_regularity_rate"].min()
)

print(
    "Maximum regularity rate:",
    mdm_kpi["mdm_regularity_rate"].max()
)

Schools: 600
Unique school IDs: 600
Regularity rates below 0%: 0
Regularity rates above 100%: 0
Missing regularity rates: 0
Minimum regularity rate: 24.242424242424242
Maximum regularity rate: 100.0


## 4. Infrastructure Condition Analysis

The infrastructure dataset contains five key school facility indicators:

- Electricity
- Drinking water
- Functional toilet
- Boundary wall
- Playground

These indicators were standardized during data cleaning so that functional
or available facilities are represented by `1`, non-functional or unavailable
facilities by `0`, and unassessed values remain missing.

Before calculating the Infrastructure Deficit Index, we examine the
distribution of each infrastructure indicator.


In [22]:
infrastructure_features = [
    "has_electricity",
    "has_drinking_water",
    "has_functional_toilet",
    "has_boundary_wall",
    "has_playground"
]

for column in infrastructure_features:
    print(f"\n{column}:")
    print(infrastructure[column].value_counts(dropna=False))


has_electricity:
has_electricity
1.0    2116
0.0     675
NaN     209
Name: count, dtype: int64

has_drinking_water:
has_drinking_water
1.0    2422
0.0     399
NaN     179
Name: count, dtype: int64

has_functional_toilet:
has_functional_toilet
1.0    2287
0.0     540
NaN     173
Name: count, dtype: int64

has_boundary_wall:
has_boundary_wall
1.0    1713
0.0    1034
NaN     253
Name: count, dtype: int64

has_playground:
has_playground
1.0    1407
0.0    1341
NaN     252
Name: count, dtype: int64


### 4.1 Infrastructure Deficit Index

Infrastructure inspections are recorded multiple times for some schools.
Therefore, infrastructure conditions are first summarized at the school level.

The Infrastructure Deficit Index measures the percentage of assessed
infrastructure indicators that are unavailable or non-functional.

Missing assessments are excluded rather than treated as failures.

The index ranges from 0% to 100%:
- 0% indicates no observed infrastructure deficit.
- 100% indicates that all assessed infrastructure indicators are deficient.

We also retain the number of assessed indicators so that schools with
substantial missing infrastructure information can be identified separately.

In [23]:
infrastructure_features = [
    "has_electricity",
    "has_drinking_water",
    "has_functional_toilet",
    "has_boundary_wall",
    "has_playground"
]

infrastructure_kpi = (
    infrastructure
    .groupby("school_id")[infrastructure_features]
    .mean()
    .reset_index()
)

infrastructure_kpi["assessed_facilities"] = (
    infrastructure_kpi[infrastructure_features]
    .notna()
    .sum(axis=1)
)

infrastructure_kpi["functional_facilities"] = (
    infrastructure_kpi[infrastructure_features]
    .sum(axis=1)
)

infrastructure_kpi["infrastructure_deficit_index"] = (
    1
    - infrastructure_kpi["functional_facilities"]
    / infrastructure_kpi["assessed_facilities"]
) * 100

infrastructure_kpi.head()

,school_id,has_electricity,has_drinking_water,has_functional_toilet,has_boundary_wall,has_playground,assessed_facilities,functional_facilities,infrastructure_deficit_index
0,SCH0001,0.500000,1.0,1.000000,0.00,0.000000,5,2.5,50.0
1,SCH0002,0.666667,1.0,1.000000,1.00,0.333333,5,4.0,20.0
2,SCH0003,1.000000,1.0,0.833333,0.60,0.666667,5,4.1,18.0
3,SCH0004,0.600000,0.8,1.000000,0.80,0.200000,5,3.4,32.0
4,SCH0005,0.250000,1.0,1.000000,0.25,0.500000,5,3.0,40.0


### 4.2 Infrastructure Deficit Index Validation

The Infrastructure Deficit Index is validated to ensure that each school has
a unique record, that the number of assessed facilities is valid, and that
the resulting deficit index remains within the expected 0–100% range.

Schools with fewer than five assessed indicators are retained because missing
assessments represent unavailable inspection information rather than confirmed
infrastructure deficiencies.

In [24]:
print("Rows in infrastructure KPI:", len(infrastructure_kpi))
print("Unique school IDs:", infrastructure_kpi["school_id"].nunique())
print("Duplicate school IDs:", infrastructure_kpi["school_id"].duplicated().sum())

print(
    "Assessed facilities below 1:",
    (infrastructure_kpi["assessed_facilities"] < 1).sum()
)

print(
    "Assessed facilities above 5:",
    (infrastructure_kpi["assessed_facilities"] > 5).sum()
)

print(
    "Deficit index below 0%:",
    (infrastructure_kpi["infrastructure_deficit_index"] < 0).sum()
)

print(
    "Deficit index above 100%:",
    (infrastructure_kpi["infrastructure_deficit_index"] > 100).sum()
)

print(
    "Missing deficit index:",
    infrastructure_kpi["infrastructure_deficit_index"].isna().sum()
)

print(
    "Minimum deficit index:",
    infrastructure_kpi["infrastructure_deficit_index"].min()
)

print(
    "Maximum deficit index:",
    infrastructure_kpi["infrastructure_deficit_index"].max()
)

Rows in infrastructure KPI: 598
Unique school IDs: 598
Duplicate school IDs: 0
Assessed facilities below 1: 0
Assessed facilities above 5: 0
Deficit index below 0%: 0
Deficit index above 100%: 0
Missing deficit index: 0
Minimum deficit index: 0.0
Maximum deficit index: 80.0


### 4.3 Infrastructure Coverage Check

The infrastructure dataset contains 598 of the 600 schools in the School
Master.

We identify the two schools without infrastructure inspection records so that
they can be handled explicitly during the final school-level join.

These schools are not assumed to have infrastructure deficiencies. Their
infrastructure status is simply unavailable in the inspection data.

In [25]:
missing_infrastructure_schools = school_master[
    ~school_master["school_id"].isin(infrastructure_kpi["school_id"])
][
    ["school_id", "school_name", "district", "block"]
]

print(
    "Schools in School Master without infrastructure records:",
    len(missing_infrastructure_schools)
)

display(missing_infrastructure_schools)

Schools in School Master without infrastructure records: 2


,school_id,school_name,district,block
273,SCH0342,Govt. Senior Secondary School Kashyap,Sangrur,Sangrur
423,SCH0374,Govt. Elementary School Yohannan,Ferozepur,Mamdot


### 4.4 Infrastructure Deficit Summary

The school-level Infrastructure Deficit Index is summarized to understand the
overall distribution of infrastructure gaps.

The summary includes the average deficit, median deficit, minimum and maximum
deficit, and the number of schools with substantial infrastructure gaps.

This provides an initial view of infrastructure conditions before examining
their relationship with attendance and student performance.

In [26]:
print("Infrastructure Deficit Index Summary:")
print(
    infrastructure_kpi["infrastructure_deficit_index"].describe()
)

print("\nSchools by deficit level:")

infrastructure_kpi["deficit_level"] = pd.cut(
    infrastructure_kpi["infrastructure_deficit_index"],
    bins=[-1, 20, 40, 60, 100],
    labels=[
        "Low (0–20%)",
        "Moderate (21–40%)",
        "High (41–60%)",
        "Very High (61–100%)"
    ]
)

print(
    infrastructure_kpi["deficit_level"]
    .value_counts()
    .sort_index()
)

Infrastructure Deficit Index Summary:
count    598.000000
mean      28.903584
std       11.232379
min        0.000000
25%       20.964286
50%       27.861111
75%       35.944444
max       80.000000
Name: infrastructure_deficit_index, dtype: float64

Schools by deficit level:
deficit_level
Low (0–20%)            137
Moderate (21–40%)      385
High (41–60%)           71
Very High (61–100%)      5
Name: count, dtype: int64


## 5. Test Score Analysis

The test-score dataset contains assessments recorded using different grading
scales, including percentages, letter grades, raw marks, and CGPA.

During data cleaning, these different formats were standardized into a common
`score_percentage` measure.

We first validate the cleaned score data before calculating school- and
district-level performance KPIs.

In [27]:
print("Test score rows:", len(test_scores))
print("Unique schools:", test_scores["school_id"].nunique())

print("\nScore percentage summary:")
print(test_scores["score_percentage"].describe())

print("\nScore percentage range:")
print(
    "Minimum:", test_scores["score_percentage"].min()
)
print(
    "Maximum:", test_scores["score_percentage"].max()
)

print("\nMissing score percentages:")
print(test_scores["score_percentage"].isna().sum())

print("\nSubjects:")
print(test_scores["subject"].value_counts())

print("\nGrading scales:")
print(test_scores["grading_scale"].value_counts())

Test score rows: 8000
Unique schools: 600

Score percentage summary:
count    8000.000000
mean       66.063937
std        15.047539
min        38.000000
25%        55.000000
50%        65.000000
75%        77.100000
max        95.000000
Name: score_percentage, dtype: float64

Score percentage range:
Minimum: 38.0
Maximum: 95.0

Missing score percentages:
0

Subjects:
subject
Math       2954
Hindi      1051
English    1022
Punjabi    1017
EVS         994
Science     962
Name: count, dtype: int64

Grading scales:
grading_scale
Raw Marks       2002
Letter Grade    1983
CGPA            1609
%                816
pct              800
Percentage       790
Name: count, dtype: int64


### 5.1 Average FLN Score by District

District information is obtained from the cleaned School Master table and
joined to the standardized test-score data using `school_id`.

The district-level FLN performance metric is calculated as the mean
standardized score percentage across assessments within each district.

This provides a comparable measure of student performance across districts.

In [28]:
test_scores_district = test_scores.merge(
    school_master[["school_id", "district"]],
    on="school_id",
    how="left",
    validate="many_to_one"
)

print("Test score rows before join:", len(test_scores))
print("Test score rows after join:", len(test_scores_district))
print("Missing districts after join:", test_scores_district["district"].isna().sum())

district_score_kpi = (
    test_scores_district
    .groupby("district")
    .agg(
        average_fln_score=("score_percentage", "mean"),
        assessments=("score_percentage", "size"),
        schools=("school_id", "nunique")
    )
    .reset_index()
    .sort_values("average_fln_score", ascending=False)
)

display(district_score_kpi)

Test score rows before join: 8000
Test score rows after join: 8000
Missing districts after join: 0


,district,average_fln_score,assessments,schools
3,Jalandhar,66.574850,835,62
2,Ferozepur,66.477603,1047,79
6,Patiala,66.415844,972,72
1,Bathinda,66.278804,1045,74
4,Ludhiana,66.156410,975,76
8,Unknown,66.116426,277,22
5,Moga,66.112154,975,71
7,Sangrur,65.253369,935,72
0,Amritsar,65.190522,939,72


### 5.2 District Coverage Validation

The district-level analysis contains an `Unknown` district category.

Rather than removing these assessments, we investigate the underlying school
records to determine why district information is unavailable.

Records with unavailable district information are retained because their test
scores remain valid observations. They will be excluded only from named
district comparisons where a known district is required.

In [30]:
unknown_district_schools = school_master[
    school_master["district"].eq("Unknown")
][
    ["school_id", "school_name", "district", "block"]
]

print(
    "Schools with Unknown district:",
    len(unknown_district_schools)
)

display(unknown_district_schools.head(30))

Schools with Unknown district: 22


,school_id,school_name,district,block
56,SCH0205,Govt. Elementary School Chander,Unknown,Ferozepur
94,SCH0453,Govt. Primary School Mahal,Unknown,Ludhiana-Ii
120,SCH0182,Govt. Middle School Hari,Unknown,Nihal Singh Wala
143,SCH0279,Govt. Senior Secondary School Narayan,Unknown,Nihal Singh Wala
152,SCH0330,Govt. Middle School Badami,Unknown,Moga-I
162,SCH0085,Govt. Elementary School Bandi,Unknown,Unknown
167,SCH0491,Govt. Primary School Raman,Unknown,Dhuri
179,SCH0519,Govt. Elementary School Palan,Unknown,Maur
225,SCH0582,Govt. Primary School Tiwari,Unknown,Rajpura
253,SCH0026,Govt. Elementary School Narayan,Unknown,Ludhiana-Ii


### 5.3 District FLN Score KPI Validation

The district-level FLN score table is validated against the original test-score
dataset.

The validation confirms that the school-to-district join did not create or
remove assessment records and that every district-level score is within the
expected 0–100% range.

Schools with an `Unknown` district are retained as an explicit category
rather than assigning a district based on inference.

In [31]:
print("Original test-score rows:", len(test_scores))
print("Rows after district join:", len(test_scores_district))

print(
    "Missing districts:",
    test_scores_district["district"].isna().sum()
)

print(
    "District KPI rows:",
    len(district_score_kpi)
)

print(
    "Scores below 0%:",
    (district_score_kpi["average_fln_score"] < 0).sum()
)

print(
    "Scores above 100%:",
    (district_score_kpi["average_fln_score"] > 100).sum()
)

print(
    "Missing district averages:",
    district_score_kpi["average_fln_score"].isna().sum()
)

Original test-score rows: 8000
Rows after district join: 8000
Missing districts: 0
District KPI rows: 9
Scores below 0%: 0
Scores above 100%: 0
Missing district averages: 0


## 6. School-Level Test Performance

To compare academic performance with attendance, MDM procurement regularity,
and infrastructure conditions, test scores are aggregated to the school level.

The school-level metric is the average standardized test score percentage
across all assessments for each school.

We also retain the number of assessments and subjects represented for each
school to provide context for the average score.

In [32]:
test_score_kpi = (
    test_scores
    .groupby("school_id")
    .agg(
        average_test_score=("score_percentage", "mean"),
        total_assessments=("score_percentage", "size"),
        subjects_assessed=("subject", "nunique")
    )
    .reset_index()
)

display(test_score_kpi.head())

,school_id,average_test_score,total_assessments,subjects_assessed
0,SCH0001,60.371875,16,6
1,SCH0002,63.460000,15,5
2,SCH0003,71.445833,12,4
3,SCH0004,66.953333,15,5
4,SCH0005,64.480769,13,5


### 6.1 School-Level Test Performance Validation

The school-level test-score KPI is validated to ensure that every school is
represented exactly once and that average scores remain within the expected
0–100% range.

The number of assessments and subjects is also checked to ensure that every
school has valid supporting assessment data.

In [33]:
print("Rows in test score KPI:", len(test_score_kpi))
print("Unique school IDs:", test_score_kpi["school_id"].nunique())
print("Duplicate school IDs:", test_score_kpi["school_id"].duplicated().sum())

print(
    "Average scores below 0%:",
    (test_score_kpi["average_test_score"] < 0).sum()
)

print(
    "Average scores above 100%:",
    (test_score_kpi["average_test_score"] > 100).sum()
)

print(
    "Missing average scores:",
    test_score_kpi["average_test_score"].isna().sum()
)

print(
    "Schools with zero assessments:",
    (test_score_kpi["total_assessments"] == 0).sum()
)

print(
    "Schools with zero subjects:",
    (test_score_kpi["subjects_assessed"] == 0).sum()
)

Rows in test score KPI: 600
Unique school IDs: 600
Duplicate school IDs: 0
Average scores below 0%: 0
Average scores above 100%: 0
Missing average scores: 0
Schools with zero assessments: 0
Schools with zero subjects: 0


## 7. Combined School-Level Analytical Dataset

The cleaned datasets have been independently aggregated to the school level.
We now combine these KPI tables into a single analytical dataset.

The School Master is used as the base table because it contains all 600
schools.

Each supporting KPI table is joined using `school_id`. Left joins are used
so that schools without infrastructure inspection records are retained
rather than being silently removed.

The resulting dataset should contain one row per school.

In [34]:
school_analysis = (
    school_master
    .merge(
        attendance_kpi,
        on="school_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        mdm_kpi,
        on="school_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        infrastructure_kpi[
            [
                "school_id",
                "assessed_facilities",
                "functional_facilities",
                "infrastructure_deficit_index"
            ]
        ],
        on="school_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        test_score_kpi,
        on="school_id",
        how="left",
        validate="one_to_one"
    )
)

print("Rows:", len(school_analysis))
print("Columns:", len(school_analysis.columns))

display(school_analysis.head())

Rows: 600
Columns: 28


,school_id,school_name,district,block,total_enrolled_students,school_type,medium,avg_attendance_rate,total_attendance_records,proxy_attendance_records,...,avg_mdm_cost,unique_vendors,unique_grain_types,mdm_regularity_rate,assessed_facilities,functional_facilities,infrastructure_deficit_index,average_test_score,total_assessments,subjects_assessed
0,SCH0050,Govt. Senior Secondary School Tank,Moga,Moga-I,424,Secondary,Punjabi,81.642308,37,2,...,2502.909091,10,7,66.666667,5.0,2.700000,46.000000,70.010526,19,5
1,SCH0583,Govt. Middle School Dara,Ferozepur,Makhu,436,Higher Secondary,Punjabi,78.002188,30,2,...,2702.689655,11,8,87.878788,5.0,3.450000,31.000000,62.793750,16,4
2,SCH0083,Govt. Primary School Dutta,Bathinda,Talwandi Sabo,196,Primary,Hindi,79.203361,37,0,...,2847.720000,9,6,75.757576,5.0,4.000000,20.000000,68.917857,14,5
3,SCH0306,Govt. Elementary School Bhardwaj,Patiala,Unknown,375,Primary,Hindi,83.958656,31,2,...,1915.888889,10,5,54.545455,5.0,3.166667,36.666667,70.341176,17,5
4,SCH0110,Govt. Primary School Goswami,Moga,Bagha Purana,398,Higher Secondary,Punjabi,79.334332,35,0,...,2296.782609,9,8,69.696970,5.0,3.500000,30.000000,61.445000,10,5


### 7.1 Combined Dataset Validation

The combined analytical dataset should contain exactly one row for every
school in the School Master.

We validate the row count, uniqueness of school IDs, missing KPI values, and
in particular the two schools that do not have infrastructure inspection
records.

Missing infrastructure values are expected for these schools and do not
represent confirmed infrastructure deficiencies.

In [35]:
print("Rows in combined dataset:", len(school_analysis))
print("Unique school IDs:", school_analysis["school_id"].nunique())
print("Duplicate school IDs:", school_analysis["school_id"].duplicated().sum())

print("\nMissing values in major KPIs:")

kpi_columns = [
    "avg_attendance_rate",
    "proxy_attendance_rate",
    "mdm_regularity_rate",
    "infrastructure_deficit_index",
    "average_test_score"
]

print(school_analysis[kpi_columns].isna().sum())

print("\nSchools missing infrastructure KPI:")
display(
    school_analysis[
        school_analysis["infrastructure_deficit_index"].isna()
    ][
        ["school_id", "school_name", "district", "infrastructure_deficit_index"]
    ]
)

Rows in combined dataset: 600
Unique school IDs: 600
Duplicate school IDs: 0

Missing values in major KPIs:
avg_attendance_rate             0
proxy_attendance_rate           0
mdm_regularity_rate             0
infrastructure_deficit_index    2
average_test_score              0
dtype: int64

Schools missing infrastructure KPI:


,school_id,school_name,district,infrastructure_deficit_index
273,SCH0342,Govt. Senior Secondary School Kashyap,Sangrur,NaN
423,SCH0374,Govt. Elementary School Yohannan,Ferozepur,NaN


## 8. MDM Regularity and Attendance Relationship

We examine whether schools with more regular recorded MDM procurement also
tend to have higher average attendance rates.

A Pearson correlation is used to measure the strength and direction of the
linear association between MDM Regularity Rate and Average Attendance Rate.

This analysis identifies an association rather than establishing that MDM
procurement causes changes in attendance.

In [36]:
mdm_attendance_corr = school_analysis[
    [
        "mdm_regularity_rate",
        "avg_attendance_rate"
    ]
].corr().iloc[0, 1]

print(
    "Correlation between MDM Regularity Rate and Average Attendance Rate:",
    round(mdm_attendance_corr, 3)
)

Correlation between MDM Regularity Rate and Average Attendance Rate: 0.003


### 8.1 MDM Regularity vs Attendance Visualization

A scatter plot is used to visually examine the relationship between MDM
Regularity Rate and Average Attendance Rate across schools.

Plotly is used so that the visualization can later be reused as an
interactive dashboard component.

In [38]:
import plotly.express as px

fig = px.scatter(
    school_analysis,
    x="mdm_regularity_rate",
    y="avg_attendance_rate",
    hover_data=["school_id", "district"],
    title="MDM Regularity vs Average Attendance",
    labels={
        "mdm_regularity_rate": "MDM Regularity Rate (%)",
        "avg_attendance_rate": "Average Attendance Rate (%)"
    }
)

fig.show()

In [ ]:
#MDM procurement regularity shows virtually no linear association with average school attendance (r = 0.003).

## 9. Infrastructure Deficit and Attendance Relationship

We examine whether schools with greater infrastructure deficits tend to have
lower average attendance rates.

A Pearson correlation is used to measure the strength and direction of the
linear association between the Infrastructure Deficit Index and Average
Attendance Rate.

The two schools without infrastructure inspection records are excluded from
this relationship analysis because their infrastructure status is unavailable.

This analysis identifies an association and does not establish that
infrastructure conditions directly cause changes in attendance.

In [39]:
infra_attendance_data = school_analysis[
    [
        "school_id",
        "infrastructure_deficit_index",
        "avg_attendance_rate"
    ]
].dropna()

infra_attendance_corr = infra_attendance_data[
    [
        "infrastructure_deficit_index",
        "avg_attendance_rate"
    ]
].corr().iloc[0, 1]

print("Schools included:", len(infra_attendance_data))
print(
    "Correlation between Infrastructure Deficit and Attendance:",
    round(infra_attendance_corr, 3)
)

Schools included: 598
Correlation between Infrastructure Deficit and Attendance: 0.002


## 10. Electricity Availability and Test Performance

The project asks whether average test scores differ between schools with and
without functional electricity.

We compare school-level average test scores across the two electricity
status groups.

Only schools with a known electricity status are included. Missing
electricity assessments are excluded from this comparison.

This analysis identifies an association between electricity availability and
test performance and does not establish a causal effect.

In [40]:
electricity_score_data = school_analysis[
    [
        "school_id",
        "has_electricity",
        "average_test_score"
    ]
].dropna(subset=["has_electricity", "average_test_score"])

electricity_score_comparison = (
    electricity_score_data
    .groupby("has_electricity")
    .agg(
        average_test_score=("average_test_score", "mean"),
        schools=("school_id", "nunique")
    )
    .reset_index()
)

electricity_score_comparison["electricity_status"] = (
    electricity_score_comparison["has_electricity"]
    .map({
        1.0: "Functional electricity",
        0.0: "No functional electricity"
    })
)

display(
    electricity_score_comparison[
        ["electricity_status", "average_test_score", "schools"]
    ]
)

KeyError: "['has_electricity'] not in index"

### 10.1 Add Electricity Status to the School-Level Dataset

The combined school-level dataset currently contains the overall
Infrastructure Deficit Index but not the individual infrastructure
indicators.

For the electricity analysis, the school-level electricity indicator is
taken from the aggregated infrastructure KPI table.

A value of 1 indicates that electricity was functional across the recorded
inspections, while 0 indicates that it was not functional. Schools with
mixed or missing electricity inspection results are retained in the dataset
but excluded from the simple functional-versus-non-functional comparison.

In [41]:
school_analysis = school_analysis.merge(
    infrastructure_kpi[
        [
            "school_id",
            "has_electricity"
        ]
    ],
    on="school_id",
    how="left",
    validate="one_to_one"
)

print("Rows after adding electricity:", len(school_analysis))
print("\nElectricity status:")
print(
    school_analysis["has_electricity"]
    .value_counts(dropna=False)
)

Rows after adding electricity: 600

Electricity status:
has_electricity
1.000000    178
0.666667     74
0.750000     58
0.800000     53
0.500000     52
0.833333     33
0.600000     28
0.333333     17
0.857143     16
0.000000     13
0.714286     11
0.250000     10
0.571429      9
0.625000      8
0.400000      7
0.777778      6
0.875000      6
0.700000      5
NaN           5
0.888889      3
0.818182      2
0.555556      2
0.900000      1
0.444444      1
0.200000      1
0.375000      1
Name: count, dtype: int64


### 10.2 Latest Infrastructure Inspection Status

Infrastructure conditions are recorded through periodic inspections, so an
average across all inspections does not necessarily represent the school's
current infrastructure status.

For comparisons involving a specific facility, the most recent available
inspection for each school is therefore used.

This provides one current-status observation per school while preserving the
historical inspection data for other analyses.

In [42]:
infrastructure_latest = (
    infrastructure
    .sort_values("date")
    .groupby("school_id")
    .tail(1)
    .copy()
)

print("Schools with latest infrastructure inspection:", len(infrastructure_latest))
print("Unique school IDs:", infrastructure_latest["school_id"].nunique())

print("\nLatest electricity status:")
print(
    infrastructure_latest["has_electricity"]
    .value_counts(dropna=False)
)

Schools with latest infrastructure inspection: 598
Unique school IDs: 598

Latest electricity status:
has_electricity
1.0    437
0.0    131
NaN     30
Name: count, dtype: int64


### 10.3 Test Scores by Latest Electricity Status

The latest available infrastructure inspection is used to classify schools
according to their electricity status.

We compare the average standardized test score between schools with
functional electricity and schools without functional electricity.

Schools with a missing electricity assessment are excluded from this
comparison.

The comparison describes an association between electricity status and test
performance; it does not establish causation.

In [43]:
electricity_test_scores = test_score_kpi.merge(
    infrastructure_latest[
        [
            "school_id",
            "has_electricity"
        ]
    ],
    on="school_id",
    how="inner",
    validate="one_to_one"
)

electricity_test_scores = electricity_test_scores.dropna(
    subset=["has_electricity"]
)

electricity_score_comparison = (
    electricity_test_scores
    .groupby("has_electricity")
    .agg(
        average_test_score=("average_test_score", "mean"),
        schools=("school_id", "nunique")
    )
    .reset_index()
)

electricity_score_comparison["electricity_status"] = (
    electricity_score_comparison["has_electricity"]
    .map({
        1.0: "Functional electricity",
        0.0: "No functional electricity"
    })
)

display(
    electricity_score_comparison[
        ["electricity_status", "average_test_score", "schools"]
    ]
)

,electricity_status,average_test_score,schools
0,No functional electricity,66.026540,131
1,Functional electricity,66.069673,437


### 10.4 Electricity and Test Score Difference

The difference in average test scores between schools with and without
functional electricity is calculated to quantify the size of the observed
gap.

A very small difference would indicate that electricity status alone does
not meaningfully distinguish average school-level test performance in this
dataset.

In [44]:
functional_score = electricity_score_comparison.loc[
    electricity_score_comparison["has_electricity"] == 1.0,
    "average_test_score"
].iloc[0]

nonfunctional_score = electricity_score_comparison.loc[
    electricity_score_comparison["has_electricity"] == 0.0,
    "average_test_score"
].iloc[0]

score_difference = functional_score - nonfunctional_score

print("Average score with functional electricity:",
      round(functional_score, 3))

print("Average score without functional electricity:",
      round(nonfunctional_score, 3))

print("Difference:",
      round(score_difference, 3),
      "percentage points")

Average score with functional electricity: 66.07
Average score without functional electricity: 66.027
Difference: 0.043 percentage points


## 11. Facility-Level Infrastructure and Attendance Analysis

The overall Infrastructure Deficit Index combines five infrastructure
indicators into a single measure. To identify which individual facilities
may be associated with attendance, each facility is analyzed separately.

The latest available infrastructure inspection is used to represent the
school's most recent infrastructure status.

For each facility, average attendance is compared between schools where the
facility is functional and schools where it is not functional.

The attendance data does not contain student gender information, so the
required functional-toilet versus female-attendance comparison cannot be
calculated directly. Overall attendance is used as the supported alternative.

These comparisons show association only and do not establish causality.

In [46]:
facility_attendance_data = (
    attendance_kpi[
        ["school_id", "avg_attendance_rate"]
    ]
    .merge(
        infrastructure_latest[
            [
                "school_id",
                "has_electricity",
                "has_drinking_water",
                "has_functional_toilet",
                "has_boundary_wall",
                "has_playground"
            ]
        ],
        on="school_id",
        how="inner",
        validate="one_to_one"
    )
)

print("Rows:", len(facility_attendance_data))
print("Unique schools:", facility_attendance_data["school_id"].nunique())
print("Duplicate school IDs:",
      facility_attendance_data["school_id"].duplicated().sum())

display(facility_attendance_data.head())

Rows: 598
Unique schools: 598
Duplicate school IDs: 0


,school_id,avg_attendance_rate,has_electricity,has_drinking_water,has_functional_toilet,has_boundary_wall,has_playground
0,SCH0001,82.288549,0.0,1.0,NaN,0.0,0.0
1,SCH0002,83.668212,1.0,1.0,1.0,1.0,1.0
2,SCH0003,80.746589,1.0,1.0,1.0,NaN,1.0
3,SCH0004,79.194911,1.0,1.0,1.0,1.0,0.0
4,SCH0005,80.676448,0.0,1.0,1.0,0.0,0.0


In [47]:
facility_columns = [
    "has_electricity",
    "has_drinking_water",
    "has_functional_toilet",
    "has_boundary_wall",
    "has_playground"
]

facility_results = []

for facility in facility_columns:

    temp = facility_attendance_data[
        ["school_id", "avg_attendance_rate", facility]
    ].dropna(subset=[facility])

    grouped = (
        temp
        .groupby(facility)
        .agg(
            average_attendance=("avg_attendance_rate", "mean"),
            schools=("school_id", "nunique")
        )
    )

    functional_attendance = (
        grouped.loc[1.0, "average_attendance"]
        if 1.0 in grouped.index else None
    )

    nonfunctional_attendance = (
        grouped.loc[0.0, "average_attendance"]
        if 0.0 in grouped.index else None
    )

    difference = (
        functional_attendance - nonfunctional_attendance
        if functional_attendance is not None
        and nonfunctional_attendance is not None
        else None
    )

    facility_results.append({
        "facility": facility,
        "functional_attendance": functional_attendance,
        "nonfunctional_attendance": nonfunctional_attendance,
        "difference_percentage_points": difference,
        "functional_schools": (
            grouped.loc[1.0, "schools"]
            if 1.0 in grouped.index else 0
        ),
        "nonfunctional_schools": (
            grouped.loc[0.0, "schools"]
            if 0.0 in grouped.index else 0
        )
    })

facility_results = pd.DataFrame(facility_results)

facility_results["facility"] = facility_results["facility"].replace({
    "has_electricity": "Electricity",
    "has_drinking_water": "Drinking Water",
    "has_functional_toilet": "Functional Toilet",
    "has_boundary_wall": "Boundary Wall",
    "has_playground": "Playground"
})

display(facility_results.round(3))

,facility,functional_attendance,nonfunctional_attendance,difference_percentage_points,functional_schools,nonfunctional_schools
0,Electricity,81.658,81.734,-0.075,437,131
1,Drinking Water,81.677,81.607,0.070,493,68
2,Functional Toilet,81.690,81.659,0.031,452,114
3,Boundary Wall,81.599,81.729,-0.130,339,208
4,Playground,81.663,81.674,-0.010,269,276


Infrastructure conditions show little association with average attendance in the available data. Differences between schools with functional and non-functional facilities are all below 0.2 percentage points.

## 12. School Risk Indicator Exploration

The previous analysis found that individual welfare factors have weak linear
relationships with attendance. Instead of forcing causal interpretations, we
now examine whether multiple warning indicators occur together at the
school level.

The candidate indicators are:

- Low average attendance
- High potential proxy attendance rate
- High attendance count anomaly rate
- Low MDM regularity
- High infrastructure deficit
- Low average test score

Before constructing a composite Student Welfare Risk Score, the distributions
of these indicators are examined so that thresholds can be chosen using the
observed data rather than arbitrary values.

In [48]:
risk_variables = [
    "avg_attendance_rate",
    "proxy_attendance_rate",
    "count_anomaly_rate",
    "mdm_regularity_rate",
    "infrastructure_deficit_index",
    "average_test_score"
]

risk_summary = school_analysis[risk_variables].describe().T[
    ["count", "mean", "25%", "50%", "75%", "min", "max"]
]

display(risk_summary.round(2))

,count,mean,25%,50%,75%,min,max
avg_attendance_rate,600.0,81.66,79.93,81.69,83.24,74.70,88.73
proxy_attendance_rate,600.0,4.94,2.78,4.55,6.90,0.00,17.86
count_anomaly_rate,600.0,4.06,0.00,3.45,6.06,0.00,22.22
mdm_regularity_rate,600.0,60.61,51.52,60.61,69.70,24.24,100.00
infrastructure_deficit_index,598.0,28.90,20.96,27.86,35.94,0.00,80.00
average_test_score,600.0,66.11,63.41,66.02,69.04,54.98,80.69


## 13. Student Welfare Risk Score

A composite Student Welfare Risk Score is created to identify schools that
show multiple warning indicators across attendance, attendance integrity,
MDM regularity, infrastructure, and learning performance.

Risk thresholds are derived from the observed distributions:

- Attendance below the 25th percentile
- Potential proxy attendance above the 75th percentile
- Attendance count anomaly rate above the 75th percentile
- MDM regularity below the 25th percentile
- Infrastructure deficit above the 75th percentile
- Average test score below the 25th percentile

Each condition contributes one risk point.

The resulting score ranges from 0 to 6. It is intended as a transparent
school-prioritization indicator rather than a validated prediction model
or causal measure.

In [49]:
risk_data = school_analysis.copy()

risk_data["low_attendance_flag"] = (
    risk_data["avg_attendance_rate"] < 79.93
).astype(int)

risk_data["high_proxy_flag"] = (
    risk_data["proxy_attendance_rate"] > 6.90
).astype(int)

risk_data["high_count_anomaly_flag"] = (
    risk_data["count_anomaly_rate"] > 6.06
).astype(int)

risk_data["low_mdm_regularity_flag"] = (
    risk_data["mdm_regularity_rate"] < 51.52
).astype(int)

risk_data["high_infrastructure_deficit_flag"] = (
    risk_data["infrastructure_deficit_index"] > 35.94
).astype(int)

risk_data["low_test_score_flag"] = (
    risk_data["average_test_score"] < 63.41
).astype(int)

risk_flags = [
    "low_attendance_flag",
    "high_proxy_flag",
    "high_count_anomaly_flag",
    "low_mdm_regularity_flag",
    "high_infrastructure_deficit_flag",
    "low_test_score_flag"
]

risk_data["student_welfare_risk_score"] = risk_data[risk_flags].sum(axis=1)

risk_data["risk_level"] = pd.cut(
    risk_data["student_welfare_risk_score"],
    bins=[-1, 1, 3, 6],
    labels=["Low", "Moderate", "High"]
)

display(
    risk_data[
        [
            "school_id",
            "district",
            "student_welfare_risk_score",
            "risk_level"
        ]
    ].head(10)
)

,school_id,district,student_welfare_risk_score,risk_level
0,SCH0050,Moga,1,Low
1,SCH0583,Ferozepur,2,Moderate
2,SCH0083,Bathinda,1,Low
3,SCH0306,Patiala,2,Moderate
4,SCH0110,Moga,2,Moderate
5,SCH0011,Patiala,2,Moderate
6,SCH0269,Amritsar,2,Moderate
7,SCH0354,Moga,1,Low
8,SCH0329,Sangrur,3,Moderate
9,SCH0534,Ferozepur,2,Moderate


## 14. Student Welfare Risk Distribution

The distribution of the Student Welfare Risk Score is examined to determine
how schools are distributed across Low, Moderate, and High risk categories.

This validation helps assess whether the composite score provides useful
differentiation between schools before it is used as a dashboard KPI.

In [50]:
risk_distribution = (
    risk_data["risk_level"]
    .value_counts()
    .reindex(["Low", "Moderate", "High"])
    .fillna(0)
    .astype(int)
)

print("Schools by risk level:")
print(risk_distribution)

print("\nPercentage of schools:")
print(
    (risk_distribution / len(risk_data) * 100)
    .round(2)
)

print("\nRisk score distribution:")
print(
    risk_data["student_welfare_risk_score"]
    .value_counts()
    .sort_index()
)

Schools by risk level:
risk_level
Low         304
Moderate    286
High         10
Name: count, dtype: int64

Percentage of schools:
risk_level
Low         50.67
Moderate    47.67
High         1.67
Name: count, dtype: float64

Risk score distribution:
student_welfare_risk_score
0     81
1    223
2    200
3     86
4      8
5      2
Name: count, dtype: int64


## 15. High-Risk School Identification

Schools with a Student Welfare Risk Score of 4 or higher are classified as
high risk.

These schools are examined in greater detail to identify the specific
warning indicators contributing to their risk classification.

The purpose is to support targeted intervention and dashboard storytelling,
not to label schools as definitively failing or unsafe.

In [51]:
high_risk_schools = (
    risk_data[
        risk_data["student_welfare_risk_score"] >= 4
    ]
    .sort_values(
        ["student_welfare_risk_score", "avg_attendance_rate"],
        ascending=[False, True]
    )
)

high_risk_columns = [
    "school_id",
    "school_name",
    "district",
    "student_welfare_risk_score",
    "risk_level",
    "avg_attendance_rate",
    "proxy_attendance_rate",
    "count_anomaly_rate",
    "mdm_regularity_rate",
    "infrastructure_deficit_index",
    "average_test_score"
]

display(
    high_risk_schools[high_risk_columns].round(2)
)

,school_id,school_name,district,student_welfare_risk_score,risk_level,avg_attendance_rate,proxy_attendance_rate,count_anomaly_rate,mdm_regularity_rate,infrastructure_deficit_index,average_test_score
494,SCH0359,Govt. Primary School Bhavsar,Moga,5,High,77.91,3.23,9.68,48.48,36.67,61.63
131,SCH0258,Govt. Primary School Kalita,Ludhiana,5,High,83.93,11.11,11.11,48.48,40.71,62.75
292,SCH0556,Govt. Middle School Dar,Amritsar,4,High,79.02,8.33,2.78,51.52,25.56,63.15
88,SCH0213,Govt. Elementary School Bhatt,Bathinda,4,High,79.92,8.33,2.78,66.67,42.00,61.59
340,SCH0009,Govt. Middle School Bakshi,Patiala,4,High,80.93,7.69,3.85,48.48,40.48,62.64
297,SCH0027,Govt. Primary School Lal,Sangrur,4,High,82.22,10.00,7.50,75.76,36.67,61.17
365,SCH0382,Govt. Primary School Lanka,Moga,4,High,82.42,7.89,7.89,51.52,38.00,72.05
429,SCH0458,Govt. Senior Secondary School Murty,Jalandhar,4,High,85.10,9.09,6.06,48.48,41.67,70.72
235,SCH0470,Govt. Elementary School Handa,Jalandhar,4,High,86.34,7.69,11.54,33.33,40.00,70.38
237,SCH0186,Govt. Senior Secondary School Pillay,Moga,4,High,88.06,9.68,9.68,84.85,41.00,62.88


## 16. High-Risk School Warning Indicators

To make the Student Welfare Risk Score interpretable, the individual warning
flags contributing to each high-risk school's score are displayed.

Each flagged indicator represents one observed risk condition:

- Low attendance
- High potential proxy attendance
- High attendance count anomalies
- Low MDM regularity
- High infrastructure deficit
- Low test performance

This allows the dashboard to show not only which schools require attention,
but also the specific areas contributing to their risk score.

In [52]:
risk_reason_columns = [
    "low_attendance_flag",
    "high_proxy_flag",
    "high_count_anomaly_flag",
    "low_mdm_regularity_flag",
    "high_infrastructure_deficit_flag",
    "low_test_score_flag"
]

risk_reason_labels = {
    "low_attendance_flag": "Low Attendance",
    "high_proxy_flag": "High Proxy Attendance",
    "high_count_anomaly_flag": "Attendance Count Anomalies",
    "low_mdm_regularity_flag": "Low MDM Regularity",
    "high_infrastructure_deficit_flag": "High Infrastructure Deficit",
    "low_test_score_flag": "Low Test Score"
}

high_risk_reasons = high_risk_schools[
    ["school_id", "school_name", "district",
     "student_welfare_risk_score"] + risk_reason_columns
].copy()

for column, label in risk_reason_labels.items():
    high_risk_reasons[label] = high_risk_reasons[column]

high_risk_reasons = high_risk_reasons.drop(
    columns=risk_reason_columns
)

display(high_risk_reasons)

,school_id,school_name,district,student_welfare_risk_score,Low Attendance,High Proxy Attendance,Attendance Count Anomalies,Low MDM Regularity,High Infrastructure Deficit,Low Test Score
494,SCH0359,Govt. Primary School Bhavsar,Moga,5,1,0,1,1,1,1
131,SCH0258,Govt. Primary School Kalita,Ludhiana,5,0,1,1,1,1,1
292,SCH0556,Govt. Middle School Dar,Amritsar,4,1,1,0,1,0,1
88,SCH0213,Govt. Elementary School Bhatt,Bathinda,4,1,1,0,0,1,1
340,SCH0009,Govt. Middle School Bakshi,Patiala,4,0,1,0,1,1,1
297,SCH0027,Govt. Primary School Lal,Sangrur,4,0,1,1,0,1,1
365,SCH0382,Govt. Primary School Lanka,Moga,4,0,1,1,1,1,0
429,SCH0458,Govt. Senior Secondary School Murty,Jalandhar,4,0,1,1,1,1,0
235,SCH0470,Govt. Elementary School Handa,Jalandhar,4,0,1,1,1,1,0
237,SCH0186,Govt. Senior Secondary School Pillay,Moga,4,0,1,1,0,1,1


## 17. Risk Score Validation

The Student Welfare Risk Score should equal the sum of its six binary
warning indicators.

This validation ensures that the composite score is internally consistent
and that no calculation errors were introduced when assigning risk levels.

In [53]:
risk_data["calculated_risk_score"] = risk_data[
    risk_flags
].sum(axis=1)

print(
    "Risk score mismatches:",
    (
        risk_data["student_welfare_risk_score"]
        != risk_data["calculated_risk_score"]
    ).sum()
)

print(
    "Minimum risk score:",
    risk_data["student_welfare_risk_score"].min()
)

print(
    "Maximum risk score:",
    risk_data["student_welfare_risk_score"].max()
)

print(
    "Missing risk scores:",
    risk_data["student_welfare_risk_score"].isna().sum()
)

Risk score mismatches: 0
Minimum risk score: 0
Maximum risk score: 5
Missing risk scores: 0


## 18. District-Level Student Welfare Risk

School-level risk scores are aggregated to the district level to identify
where high-risk schools are concentrated.

For each district, we calculate:

- Total schools
- Average Student Welfare Risk Score
- Number of high-risk schools
- Percentage of schools classified as high risk

District-level results can help education administrators prioritize areas
for further investigation.

These results indicate concentration of observed risk indicators and do not
imply that a district is responsible for the underlying conditions.

In [54]:
district_risk_kpi = (
    risk_data
    .groupby("district")
    .agg(
        total_schools=("school_id", "nunique"),
        average_risk_score=("student_welfare_risk_score", "mean"),
        high_risk_schools=(
            "student_welfare_risk_score",
            lambda x: (x >= 4).sum()
        )
    )
    .reset_index()
)

district_risk_kpi["high_risk_percentage"] = (
    district_risk_kpi["high_risk_schools"]
    / district_risk_kpi["total_schools"]
    * 100
)

district_risk_kpi = district_risk_kpi.sort_values(
    "high_risk_percentage",
    ascending=False
)

display(district_risk_kpi.round(2))

,district,total_schools,average_risk_score,high_risk_schools,high_risk_percentage
5,Moga,71,1.70,3,4.23
3,Jalandhar,62,1.58,2,3.23
0,Amritsar,72,1.42,1,1.39
6,Patiala,72,1.43,1,1.39
7,Sangrur,72,1.58,1,1.39
1,Bathinda,74,1.54,1,1.35
4,Ludhiana,76,1.59,1,1.32
2,Ferozepur,79,1.48,0,0.00
8,Unknown,22,1.50,0,0.00


## 19. Monthly Attendance Trend

Attendance records are aggregated by month to examine how average attendance
changes over time.

Monthly aggregation helps identify:

- periods of unusually low attendance,
- periods of unusually high attendance,
- changes in attendance patterns over time.

The analysis uses the cleaned attendance rate and does not remove records
flagged as potential proxy attendance or attendance-count anomalies. This
preserves the original evidence while allowing the trend to reflect the
cleaned dataset.

In [55]:
# Make sure date is in datetime format
attendance["date"] = pd.to_datetime(attendance["date"])

monthly_attendance = (
    attendance
    .groupby(attendance["date"].dt.to_period("M"))
    .agg(
        average_attendance_rate=("attendance_rate", "mean"),
        attendance_records=("school_id", "size"),
        schools=("school_id", "nunique")
    )
    .reset_index()
)

# Convert Period to timestamp for easier plotting later
monthly_attendance["month"] = (
    monthly_attendance["date"]
    .dt.to_timestamp()
)

monthly_attendance = monthly_attendance.drop(columns="date")

display(monthly_attendance.round(2))

C:\Users\helle\AppData\Local\Temp\ipykernel_11652\587005958.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(monthly_attendance.round(2))


,average_attendance_rate,attendance_records,schools,month
0,81.87,162,140,2025-01-01
1,82.70,188,163,2025-02-01
2,82.32,178,149,2025-03-01
3,82.05,1615,555,2025-04-01
4,80.84,1603,568,2025-05-01
5,82.14,1582,551,2025-06-01
6,81.58,1666,564,2025-07-01
7,81.00,1691,570,2025-08-01
8,81.85,1570,553,2025-09-01
9,81.47,1659,554,2025-10-01


## 20. Monthly Attendance Coverage Validation

Monthly attendance trends are interpreted alongside school coverage.

The dataset contains 600 schools in the School Master table, but attendance
coverage varies substantially by month. Therefore, months with very low
school coverage should not be interpreted as representative statewide
attendance trends.

School coverage is calculated as the percentage of the 600 master schools
with at least one attendance record in each month.

In [56]:
monthly_attendance["school_coverage_percentage"] = (
    monthly_attendance["schools"] / school_master["school_id"].nunique()
) * 100

display(
    monthly_attendance[
        [
            "month",
            "average_attendance_rate",
            "attendance_records",
            "schools",
            "school_coverage_percentage"
        ]
    ].round(2)
)

C:\Users\helle\AppData\Local\Temp\ipykernel_11652\3254206509.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,month,average_attendance_rate,attendance_records,schools,school_coverage_percentage
0,2025-01-01,81.87,162,140,23.33
1,2025-02-01,82.70,188,163,27.17
2,2025-03-01,82.32,178,149,24.83
3,2025-04-01,82.05,1615,555,92.50
4,2025-05-01,80.84,1603,568,94.67
5,2025-06-01,82.14,1582,551,91.83
6,2025-07-01,81.58,1666,564,94.00
7,2025-08-01,81.00,1691,570,95.00
8,2025-09-01,81.85,1570,553,92.17
9,2025-10-01,81.47,1659,554,92.33


## 21. Reliable Monthly Attendance Trend

Because attendance coverage varies substantially across the dataset, the
primary attendance trend is restricted to months where at least 90% of
schools have attendance records.

This provides a more representative comparison across months while
preserving all cleaned attendance records in the underlying dataset.

Months below the 90% coverage threshold are excluded from the primary trend
because their attendance averages are not sufficiently representative of
the full school population.

In [57]:
reliable_monthly_attendance = (
    monthly_attendance[
        monthly_attendance["school_coverage_percentage"] >= 90
    ]
    .copy()
    .sort_values("month")
)

print(
    "Reliable months:",
    len(reliable_monthly_attendance)
)

print(
    "Date range:",
    reliable_monthly_attendance["month"].min().strftime("%Y-%m"),
    "to",
    reliable_monthly_attendance["month"].max().strftime("%Y-%m")
)

print(
    "Minimum school coverage:",
    round(
        reliable_monthly_attendance["school_coverage_percentage"].min(),
        2
    ),
    "%"
)

display(
    reliable_monthly_attendance[
        [
            "month",
            "average_attendance_rate",
            "attendance_records",
            "schools",
            "school_coverage_percentage"
        ]
    ].round(2)
)

Reliable months: 12
Date range: 2025-04 to 2026-03
Minimum school coverage: 91.17 %


C:\Users\helle\AppData\Local\Temp\ipykernel_11652\1980177091.py:39: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,month,average_attendance_rate,attendance_records,schools,school_coverage_percentage
3,2025-04-01,82.05,1615,555,92.50
4,2025-05-01,80.84,1603,568,94.67
5,2025-06-01,82.14,1582,551,91.83
6,2025-07-01,81.58,1666,564,94.00
7,2025-08-01,81.00,1691,570,95.00
8,2025-09-01,81.85,1570,553,92.17
9,2025-10-01,81.47,1659,554,92.33
10,2025-11-01,81.50,1565,558,93.00
11,2025-12-01,82.40,1627,559,93.17
12,2026-01-01,82.42,1509,557,92.83


## 22. Reliable Monthly Attendance Trend

The reliable attendance period from April 2025 to March 2026 is visualized
to identify meaningful changes in average attendance over time.

Only months with at least 90% school coverage are included in this primary
trend, ensuring that changes are based on broadly representative school
coverage.

In [58]:
import plotly.express as px

fig = px.line(
    reliable_monthly_attendance,
    x="month",
    y="average_attendance_rate",
    markers=True,
    title="Monthly Average Attendance Rate",
    labels={
        "month": "Month",
        "average_attendance_rate": "Average Attendance Rate (%)"
    }
)

fig.update_yaxes(
    range=[
        reliable_monthly_attendance["average_attendance_rate"].min() - 1,
        reliable_monthly_attendance["average_attendance_rate"].max() + 1
    ]
)

fig.show()

## 23. Attendance Trend Summary

The reliable monthly attendance period is summarized using its highest and
lowest observed months and the change between the first and last reliable
months.

This provides precise values that can be used in the dashboard narrative
alongside the monthly attendance trend.

In [59]:
highest_month = reliable_monthly_attendance.loc[
    reliable_monthly_attendance["average_attendance_rate"].idxmax()
]

lowest_month = reliable_monthly_attendance.loc[
    reliable_monthly_attendance["average_attendance_rate"].idxmin()
]

first_month = reliable_monthly_attendance.iloc[0]
last_month = reliable_monthly_attendance.iloc[-1]

overall_change = (
    last_month["average_attendance_rate"]
    - first_month["average_attendance_rate"]
)

print(
    "Highest attendance:",
    highest_month["month"].strftime("%B %Y"),
    "-",
    round(highest_month["average_attendance_rate"], 2),
    "%"
)

print(
    "Lowest attendance:",
    lowest_month["month"].strftime("%B %Y"),
    "-",
    round(lowest_month["average_attendance_rate"], 2),
    "%"
)

print(
    "Change from first to last reliable month:",
    round(overall_change, 2),
    "percentage points"
)

Highest attendance: January 2026 - 82.42 %
Lowest attendance: May 2025 - 80.84 %
Change from first to last reliable month: -0.8 percentage points


## 24. District-Level Attendance Performance

Attendance is aggregated by district using the cleaned school-level attendance
data and joined with the School Master district information.

The district comparison identifies areas with relatively higher and lower
average attendance.

The results describe observed differences between districts and do not imply
that district-level factors cause the differences.

In [60]:
district_attendance_data = attendance_kpi.merge(
    school_master[["school_id", "district"]],
    on="school_id",
    how="left",
    validate="one_to_one"
)

district_attendance_kpi = (
    district_attendance_data
    .groupby("district")
    .agg(
        average_attendance=("avg_attendance_rate", "mean"),
        schools=("school_id", "nunique"),
        attendance_records=("total_attendance_records", "sum"),
        average_proxy_rate=("proxy_attendance_rate", "mean"),
        average_anomaly_rate=("count_anomaly_rate", "mean")
    )
    .reset_index()
    .sort_values("average_attendance", ascending=False)
)

display(
    district_attendance_kpi.round(2)
)

,district,average_attendance,schools,attendance_records,average_proxy_rate,average_anomaly_rate
2,Ferozepur,81.91,79,2600,5.42,4.94
6,Patiala,81.74,72,2299,4.85,4.41
7,Sangrur,81.74,72,2360,5.07,3.99
8,Unknown,81.72,22,709,4.79,3.51
4,Ludhiana,81.72,76,2514,5.59,4.02
0,Amritsar,81.68,72,2325,4.72,3.61
5,Moga,81.62,71,2309,4.70,3.75
1,Bathinda,81.59,74,2423,4.74,4.03
3,Jalandhar,81.14,62,2056,4.27,3.81


## 25. District-Level MDM Procurement Analysis

MDM procurement is aggregated to the district level to examine differences
in procurement volume, expenditure, and procurement regularity.

Because the supplied MDM data does not contain food consumption or wastage
measurements, actual MDM utilization and wastage cannot be calculated.

Instead, the analysis focuses on defensible procurement indicators:

- Total quantity procured in KG
- Total procurement cost
- Procurement cost per KG
- Average MDM regularity rate

These indicators describe procurement patterns and should not be interpreted
as direct measures of food consumption or wastage.

In [61]:
district_mdm_data = mdm_kpi.merge(
    school_master[["school_id", "district"]],
    on="school_id",
    how="left",
    validate="one_to_one"
)

district_mdm_kpi = (
    district_mdm_data
    .groupby("district")
    .agg(
        total_quantity_kg=("total_mdm_quantity_kg", "sum"),
        total_mdm_cost=("total_mdm_cost", "sum"),
        average_mdm_regularity=("mdm_regularity_rate", "mean"),
        schools=("school_id", "nunique")
    )
    .reset_index()
)

district_mdm_kpi["cost_per_kg"] = (
    district_mdm_kpi["total_mdm_cost"]
    / district_mdm_kpi["total_quantity_kg"]
)

district_mdm_kpi = district_mdm_kpi.sort_values(
    "total_mdm_cost",
    ascending=False
)

display(
    district_mdm_kpi.round(2)
)

,district,total_quantity_kg,total_mdm_cost,average_mdm_regularity,schools,cost_per_kg
2,Ferozepur,55833.9,3582502.0,62.87,79,64.16
4,Ludhiana,52447.7,3452239.0,61.72,76,65.82
6,Patiala,49439.1,3266207.0,61.20,72,66.07
0,Amritsar,49455.5,3200133.0,61.15,72,64.71
7,Sangrur,48709.6,3196558.0,60.19,72,65.62
1,Bathinda,50058.4,3195500.0,60.16,74,63.84
5,Moga,47583.0,3098938.0,59.97,71,65.13
3,Jalandhar,40973.9,2694868.0,58.90,62,65.77
8,Unknown,13244.4,873317.0,54.68,22,65.94


## 26. Vendor-Level MDM Procurement Analysis

MDM procurement is aggregated by vendor to identify major suppliers and
compare their procurement volume and expenditure.

The analysis calculates:

- Total quantity supplied in KG
- Total procurement cost
- Number of procurement records
- Number of schools served
- Average cost per KG

Vendor concentration can help identify major suppliers for administrative
review.

This analysis describes procurement patterns and does not by itself indicate
vendor performance, quality, or misconduct.

In [62]:
vendor_mdm_kpi = (
    mdm
    .groupby("vendor_name")
    .agg(
        total_quantity_kg=("quantity", "sum"),
        total_cost=("total_cost", "sum"),
        procurement_records=("procurement_id", "nunique"),
        schools_served=("school_id", "nunique")
    )
    .reset_index()
)

vendor_mdm_kpi["cost_per_kg"] = (
    vendor_mdm_kpi["total_cost"]
    / vendor_mdm_kpi["total_quantity_kg"]
)

vendor_mdm_kpi = vendor_mdm_kpi.sort_values(
    "total_cost",
    ascending=False
)

display(
    vendor_mdm_kpi.round(2)
)

,vendor_name,total_quantity_kg,total_cost,procurement_records,schools_served,cost_per_kg
3,Sharma Traders,65844.6,4289785.0,1940,576,65.15
1,Kumar General Store,35014.5,2354591.0,1024,496,67.25
7,kumar & co.,35498.5,2314741.0,1043,497,65.21
4,Singh Brothers,36071.5,2280619.0,1052,500,63.22
6,goyal mill,33480.6,2279290.0,987,494,68.08
2,Kumar Supplies,34722.3,2268812.0,1012,505,65.34
8,s. agro works,33767.5,2185168.0,996,494,64.71
10,singh agro,33717.5,2181212.0,1001,487,64.69
0,Goyal Rice Mill,32394.3,2179204.0,958,482,67.27
5,goyal enterprises,33649.3,2126258.0,994,483,63.19


## 27. Vendor Name Standardization

Vendor names are standardized for consistent reporting by removing
unnecessary whitespace and normalizing capitalization.

Potentially different vendor entities are not merged automatically. For
example, "Goyal Mill" and "Goyal Rice Mill" are retained as separate vendors
unless the source data provides evidence that they represent the same entity.

This conservative approach prevents incorrect aggregation of procurement
costs or quantities.

In [63]:
mdm["vendor_name"] = (
    mdm["vendor_name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

vendor_mdm_kpi = (
    mdm
    .groupby("vendor_name")
    .agg(
        total_quantity_kg=("quantity", "sum"),
        total_cost=("total_cost", "sum"),
        procurement_records=("procurement_id", "nunique"),
        schools_served=("school_id", "nunique")
    )
    .reset_index()
)

vendor_mdm_kpi["cost_per_kg"] = (
    vendor_mdm_kpi["total_cost"]
    / vendor_mdm_kpi["total_quantity_kg"]
)

vendor_mdm_kpi = vendor_mdm_kpi.sort_values(
    "total_cost",
    ascending=False
)

print("Unique vendors:", mdm["vendor_name"].nunique())

display(
    vendor_mdm_kpi.round(2)
)

Unique vendors: 11


,vendor_name,total_quantity_kg,total_cost,procurement_records,schools_served,cost_per_kg
8,Sharma Traders,65844.6,4289785.0,1940,576,65.15
4,Kumar General Store,35014.5,2354591.0,1024,496,67.25
3,Kumar & Co.,35498.5,2314741.0,1043,497,65.21
10,Singh Brothers,36071.5,2280619.0,1052,500,63.22
1,Goyal Mill,33480.6,2279290.0,987,494,68.08
5,Kumar Supplies,34722.3,2268812.0,1012,505,65.34
6,S. Agro Works,33767.5,2185168.0,996,494,64.71
9,Singh Agro,33717.5,2181212.0,1001,487,64.69
2,Goyal Rice Mill,32394.3,2179204.0,958,482,67.27
0,Goyal Enterprises,33649.3,2126258.0,994,483,63.19


## 28. Vendor Procurement Concentration

Vendor-level procurement is examined to determine how concentrated MDM
procurement is among the largest suppliers.

The share of total procurement quantity and total procurement cost accounted
for by the top vendors is calculated.

This helps identify whether procurement is broadly distributed across vendors
or heavily concentrated among a small number of suppliers.

A high concentration does not by itself indicate a procurement problem; it
only identifies where procurement volume is concentrated.

In [64]:
vendor_concentration = vendor_mdm_kpi.copy()

total_quantity = vendor_concentration["total_quantity_kg"].sum()
total_cost = vendor_concentration["total_cost"].sum()

vendor_concentration["quantity_share"] = (
    vendor_concentration["total_quantity_kg"]
    / total_quantity
    * 100
)

vendor_concentration["cost_share"] = (
    vendor_concentration["total_cost"]
    / total_cost
    * 100
)

vendor_concentration = vendor_concentration.sort_values(
    "total_cost",
    ascending=False
)

top_3_quantity_share = vendor_concentration.head(3)["quantity_share"].sum()
top_3_cost_share = vendor_concentration.head(3)["cost_share"].sum()

top_5_quantity_share = vendor_concentration.head(5)["quantity_share"].sum()
top_5_cost_share = vendor_concentration.head(5)["cost_share"].sum()

print(
    "Top 3 vendors - quantity share:",
    round(top_3_quantity_share, 2),
    "%"
)

print(
    "Top 3 vendors - cost share:",
    round(top_3_cost_share, 2),
    "%"
)

print(
    "Top 5 vendors - quantity share:",
    round(top_5_quantity_share, 2),
    "%"
)

print(
    "Top 5 vendors - cost share:",
    round(top_5_cost_share, 2),
    "%"
)

display(
    vendor_concentration[
        [
            "vendor_name",
            "total_quantity_kg",
            "total_cost",
            "quantity_share",
            "cost_share",
            "cost_per_kg"
        ]
    ].round(2)
)

Top 3 vendors - quantity share: 33.44 %
Top 3 vendors - cost share: 33.73 %
Top 5 vendors - quantity share: 50.5 %
Top 5 vendors - cost share: 50.9 %


,vendor_name,total_quantity_kg,total_cost,quantity_share,cost_share,cost_per_kg
8,Sharma Traders,65844.6,4289785.0,16.15,16.15,65.15
4,Kumar General Store,35014.5,2354591.0,8.59,8.87,67.25
3,Kumar & Co.,35498.5,2314741.0,8.71,8.72,65.21
10,Singh Brothers,36071.5,2280619.0,8.85,8.59,63.22
1,Goyal Mill,33480.6,2279290.0,8.21,8.58,68.08
5,Kumar Supplies,34722.3,2268812.0,8.52,8.54,65.34
6,S. Agro Works,33767.5,2185168.0,8.28,8.23,64.71
9,Singh Agro,33717.5,2181212.0,8.27,8.21,64.69
2,Goyal Rice Mill,32394.3,2179204.0,7.94,8.20,67.27
0,Goyal Enterprises,33649.3,2126258.0,8.25,8.01,63.19


## 29. MDM Payment Status Analysis

MDM procurement costs are analyzed by payment status to understand the
distribution of procurement expenditure across paid, cleared, pending, and
due transactions.

This provides a financial monitoring indicator for the dashboard.

Payment status is treated as a reporting field rather than evidence of
financial irregularity.

In [65]:
payment_status_kpi = (
    mdm
    .groupby("payment_status")
    .agg(
        procurement_records=("procurement_id", "nunique"),
        total_quantity_kg=("quantity", "sum"),
        total_cost=("total_cost", "sum")
    )
    .reset_index()
)

total_mdm_cost = payment_status_kpi["total_cost"].sum()

payment_status_kpi["cost_share"] = (
    payment_status_kpi["total_cost"]
    / total_mdm_cost
    * 100
)

payment_status_kpi = payment_status_kpi.sort_values(
    "total_cost",
    ascending=False
)

display(
    payment_status_kpi.round(2)
)

,payment_status,procurement_records,total_quantity_kg,total_cost,cost_share
2,Paid,5165,175526.9,11356845.0,42.76
3,Pending,3388,114849.3,7453707.0,28.06
1,Due,1787,60721.0,4043379.0,15.22
0,Cleared,1660,56648.3,3706331.0,13.95


## 30. Final School-Level Analytical Dataset

The cleaned datasets have been aggregated to the school level and combined
using the standardized school ID.

This table is designed as the main analytical dataset for dashboarding.

Each row represents one school, while the columns contain school
characteristics, attendance indicators, MDM indicators, infrastructure
conditions, test performance, and welfare-risk measures.

Before exporting the final table, we inspect its structure to ensure that
the required dashboard metrics are available and that no unintended
row multiplication has occurred.

In [66]:
print("Rows:", len(school_analysis))
print("Columns:", len(school_analysis.columns))

print("\nColumns:")
for column in school_analysis.columns:
    print("-", column)

print("\nDuplicate school IDs:",
      school_analysis["school_id"].duplicated().sum())

print("\nMissing values:")
display(
    school_analysis.isna().sum()
    .sort_values(ascending=False)
)

Rows: 600
Columns: 29

Columns:
- school_id
- school_name
- district
- block
- total_enrolled_students
- school_type
- medium
- avg_attendance_rate
- total_attendance_records
- proxy_attendance_records
- count_anomaly_records
- avg_total_students
- avg_present_students
- proxy_attendance_rate
- count_anomaly_rate
- total_mdm_quantity_kg
- total_mdm_records
- total_mdm_cost
- avg_mdm_cost
- unique_vendors
- unique_grain_types
- mdm_regularity_rate
- assessed_facilities
- functional_facilities
- infrastructure_deficit_index
- average_test_score
- total_assessments
- subjects_assessed
- has_electricity

Duplicate school IDs: 0

Missing values:


has_electricity                 5
infrastructure_deficit_index    2
assessed_facilities             2
functional_facilities           2
school_id                       0
school_type                     0
medium                          0
avg_attendance_rate             0
total_attendance_records        0
school_name                     0
district                        0
block                           0
total_enrolled_students         0
avg_present_students            0
avg_total_students              0
count_anomaly_records           0
proxy_attendance_records        0
proxy_attendance_rate           0
count_anomaly_rate              0
total_mdm_quantity_kg           0
total_mdm_records               0
unique_grain_types              0
unique_vendors                  0
avg_mdm_cost                    0
total_mdm_cost                  0
mdm_regularity_rate             0
average_test_score              0
total_assessments               0
subjects_assessed               0
dtype: int64

## 31. Final Dataset Missing-Value Check

Before creating the final dashboard dataset, we check which fields contain
missing values.

Missing values are not automatically treated as errors. Some are expected
because infrastructure inspections are not available for every school, and
some infrastructure fields may be missing within an inspection.

The purpose of this check is to distinguish expected missingness from
unexpected missingness before the dataset is exported.

In [67]:
missing_summary = (
    school_analysis
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_values"]

missing_summary = (
    missing_summary[
        missing_summary["missing_values"] > 0
    ]
    .sort_values("missing_values", ascending=False)
)

display(missing_summary)

,column,missing_values
28,has_electricity,5
22,assessed_facilities,2
23,functional_facilities,2
24,infrastructure_deficit_index,2


## 32. Final Dashboard Dataset

A compact school-level analytical dataset is created for dashboarding.

Each row represents exactly one school.

The dataset combines:
- School profile and location
- Attendance performance
- Attendance integrity indicators
- MDM procurement and regularity
- Infrastructure condition
- Student test performance
- Composite student welfare risk

Intermediate calculation fields that are not required for dashboard analysis
are excluded.

Infrastructure values that could not be assessed remain missing rather than
being converted to a negative condition.